In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Official NIRF 2025 Engineering page
url = "https://www.nirfindia.org/Rankings/2025/EngineeringRanking.html"

# Identify our educational project
headers = {
    "User-Agent": "Mozilla/5.0 (educational NIRF research project)"
}

# Fetch webpage
response = requests.get(
    url,
    headers=headers,
    timeout=30
)

# Check whether request was successful
print("Status Code:", response.status_code)

# Parse webpage
soup = BeautifulSoup(response.text, "html.parser")

# Display webpage title
print("Page Title:", soup.title.get_text(strip=True))

# Identify HTML tables
tables = soup.find_all("table")

print("Number of tables found:", len(tables))

Status Code: 200
Page Title: MoE, National Institute Ranking Framework (NIRF)
Number of tables found: 101


In [2]:
# Inspect the structure of all HTML tables

for index, table in enumerate(tables):

    rows = table.find_all("tr")

    # Extract the first row
    if rows:
        first_row = rows[0].get_text(" | ", strip=True)
    else:
        first_row = "No data"

    print(
        f"Table {index} | "
        f"Rows: {len(rows)} | "
        f"First row: {first_row[:150]}"
    )

Table 0 | Rows: 301 | First row: Institute ID | Name | City | State | Score | Rank
Table 1 | Rows: 2 | First row: TLR (100) | RPC (100) | GO (100) | OI (100) | PERCEPTION (100)
Table 2 | Rows: 2 | First row: TLR (100) | RPC (100) | GO (100) | OI (100) | PERCEPTION (100)
Table 3 | Rows: 2 | First row: TLR (100) | RPC (100) | GO (100) | OI (100) | PERCEPTION (100)
Table 4 | Rows: 2 | First row: TLR (100) | RPC (100) | GO (100) | OI (100) | PERCEPTION (100)
Table 5 | Rows: 2 | First row: TLR (100) | RPC (100) | GO (100) | OI (100) | PERCEPTION (100)
Table 6 | Rows: 2 | First row: TLR (100) | RPC (100) | GO (100) | OI (100) | PERCEPTION (100)
Table 7 | Rows: 2 | First row: TLR (100) | RPC (100) | GO (100) | OI (100) | PERCEPTION (100)
Table 8 | Rows: 2 | First row: TLR (100) | RPC (100) | GO (100) | OI (100) | PERCEPTION (100)
Table 9 | Rows: 2 | First row: TLR (100) | RPC (100) | GO (100) | OI (100) | PERCEPTION (100)
Table 10 | Rows: 2 | First row: TLR (100) | RPC (100) | GO (100) | OI (

In [3]:
from io import StringIO

# Convert HTML tables into Pandas DataFrames
dataframes = pd.read_html(StringIO(response.text))

print("Total DataFrames:", len(dataframes))

# Display dimensions of the first 20 tables
for index, df in enumerate(dataframes[:20]):

    print(f"\nTable {index}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(2))

Total DataFrames: 101

Table 0
Shape: (100, 6)
Columns: ['Institute ID', 'Name', 'City', 'State', 'Score', 'Rank']
  Institute ID                                               Name       City  \
0  IR-E-U-0456  Indian Institute of Technology MadrasMore Deta...    Chennai   
1  IR-E-I-1074  Indian Institute of Technology DelhiMore Detai...  New Delhi   

        State  Score  Rank  
0  Tamil Nadu  88.72     1  
1       Delhi  85.74     2  

Table 1
Shape: (1, 5)
Columns: ['TLR (100)', 'RPC (100)', 'GO (100)', 'OI (100)', 'PERCEPTION (100)']
   TLR (100)  RPC (100)  GO (100)  OI (100)  PERCEPTION (100)
0       95.7      90.74     82.29     63.25             100.0

Table 2
Shape: (1, 5)
Columns: ['TLR (100)', 'RPC (100)', 'GO (100)', 'OI (100)', 'PERCEPTION (100)']
   TLR (100)  RPC (100)  GO (100)  OI (100)  PERCEPTION (100)
0      85.89      93.47     81.67     62.01             93.97

Table 3
Shape: (1, 5)
Columns: ['TLR (100)', 'RPC (100)', 'GO (100)', 'OI (100)', 'PERCEPTION (100)']


In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

CATEGORIES = [
    "Overall", "University", "College", "Research", "Engineering",
    "Management", "Pharmacy", "Medical", "Dental", "Law",
    "Architecture", "Agriculture", "Innovation", "OPENUNIVERSITY",
    "SKILLUNIVERSITY", "STATEPUBLICUNIVERSITY", "SDGInstitutions"
]

all_data = []

for year in range(2017, 2026):
    for category in CATEGORIES:
        url = f"https://www.nirfindia.org/Rankings/{year}/{category}Ranking.html"
        
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
        except Exception as e:
            print(f"Error fetching {url}: {e}")
            continue

        if response.status_code != 200:
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        table = soup.find("table", id="tbl_overall")
        if not table:
            continue

        rows = table.select("tbody > tr") or table.find_all("tr", recursive=False)
        count_before = len(all_data)

        for row in rows:
            # Skip any row that is inside the hidden parameter sub-table
            if row.find_parent("table") != table:
                continue

            tds = row.find_all("td", recursive=False)
            if len(tds) < 4:
                continue

            cols = [list(td.stripped_strings)[0] if list(td.stripped_strings) else "" for td in tds]

            # Double check: Every real college row has an Institute_ID starting with 'IR'
            if not cols[0].startswith("IR"):
                continue

            if len(cols) == 6:
                inst_id, name, city, state, score, rank = cols
            elif len(cols) == 5:
                inst_id, name, city, state, rank = cols
                score = None
            else:
                continue

            # Extract the 5 parameters from the nested sub-table inside the Name cell (tds[1])
            sub_table = tds[1].find("table") if len(tds) > 1 else None
            params = [td.get_text(strip=True) for td in sub_table.find_all("td")] if sub_table else []
            tlr, rpc, go, oi, perception = params[:5] if len(params) >= 5 else [None] * 5

            all_data.append({
                "Year": year,
                "Category": category,
                "Institute_ID": inst_id,
                "Name": name,
                "City": city,
                "State": state,
                "Score": score,
                "Rank": rank,
                "TLR": tlr,
                "RPC": rpc,
                "GO": go,
                "OI": oi,
                "Perception": perception
            })

        rows_added = len(all_data) - count_before
        if rows_added > 0:
            print(f"Scraped: {year} - {category:<22} ({rows_added} rows)")

# Save everything to a single CSV
df = pd.DataFrame(all_data)
df.to_csv("nirf_2017_2025_clean.csv", index=False, encoding="utf-8-sig")
print(f"\nDone! Saved {len(df)} total rows.")


Scraped: 2017 - Overall                (100 rows)
Scraped: 2017 - University             (100 rows)
Scraped: 2017 - College                (100 rows)
Scraped: 2017 - Engineering            (100 rows)
Scraped: 2017 - Management             (50 rows)
Scraped: 2017 - Pharmacy               (50 rows)
Scraped: 2018 - Overall                (100 rows)
Scraped: 2018 - University             (100 rows)
Scraped: 2018 - College                (100 rows)
Scraped: 2018 - Engineering            (100 rows)
Scraped: 2018 - Management             (50 rows)
Scraped: 2018 - Pharmacy               (50 rows)
Scraped: 2018 - Medical                (25 rows)
Scraped: 2018 - Law                    (10 rows)
Scraped: 2018 - Architecture           (10 rows)
Scraped: 2019 - Overall                (100 rows)
Scraped: 2019 - University             (100 rows)
Scraped: 2019 - College                (100 rows)
Scraped: 2019 - Engineering            (200 rows)
Scraped: 2019 - Management             (75 rows)
Scraped: